In [1]:
import os
import pandas as pd
import boto3
from tqdm import tqdm

try:
    import kagglehub
except:
    ! pip install kagglehub
    import kagglehub

try:
    import pyarrow
except:
    ! pip install pyarrow
    import pyarrow

## Helper Classes

In [2]:
class DataCollector:
    def __init__(self, str_bucket, str_dirname_output):
        self.str_bucket = str_bucket
        self.str_dirname_output = str_dirname_output
        self.s3_client = boto3.client('s3')
        self.df_data = None

    def download_from_kaggle(self):
        """Download PaySim dataset from Kaggle using kagglehub."""
        print("Downloading PaySim dataset from Kaggle...")
        str_path = kagglehub.dataset_download('ealaxi/paysim1')
        print(f"Dataset downloaded to: {str_path}")
        
        # Find the CSV file
        list_files = [f for f in os.listdir(str_path) if f.endswith('.csv')]
        if not list_files:
            raise FileNotFoundError("No CSV file found in downloaded dataset")
        
        str_csv_path = os.path.join(str_path, list_files[0])
        print(f"Loading CSV: {str_csv_path}")
        self.df_data = pd.read_csv(str_csv_path)
        return self.df_data

    def clean_data(self):
        """Perform basic data cleaning."""
        if self.df_data is None:
            raise ValueError("Data not loaded. Call download_from_kaggle() first.")
        
        print("\nCleaning data...")
        
        # Check for nulls
        int_nulls = self.df_data.isnull().sum().sum()
        print(f"Total null values: {int_nulls}")
        
        # Remove any rows with nulls
        self.df_data = self.df_data.dropna()
        
        # Ensure correct dtypes
        self.df_data['step'] = self.df_data['step'].astype('int64')
        self.df_data['amount'] = self.df_data['amount'].astype('float64')
        self.df_data['oldbalanceOrg'] = self.df_data['oldbalanceOrg'].astype('float64')
        self.df_data['newbalanceOrig'] = self.df_data['newbalanceOrig'].astype('float64')
        self.df_data['oldbalanceDest'] = self.df_data['oldbalanceDest'].astype('float64')
        self.df_data['newbalanceDest'] = self.df_data['newbalanceDest'].astype('float64')
        self.df_data['isFraud'] = self.df_data['isFraud'].astype('int64')
        self.df_data['isFlaggedFraud'] = self.df_data['isFlaggedFraud'].astype('int64')
        
        print(f"Data shape after cleaning: {self.df_data.shape}")
        return self.df_data

    def get_summary_stats(self):
        """Compute and print summary statistics."""
        if self.df_data is None:
            raise ValueError("Data not loaded.")
        
        print("\n=== SUMMARY STATISTICS ===")
        print(f"Total records: {len(self.df_data):,}")
        print(f"Total features: {len(self.df_data.columns)}")
        print(f"\nFraud statistics:")
        print(f"  Non-fraud: {(self.df_data['isFraud'] == 0).sum():,} ({(self.df_data['isFraud'] == 0).sum() / len(self.df_data) * 100:.2f}%)")
        print(f"  Fraud: {(self.df_data['isFraud'] == 1).sum():,} ({(self.df_data['isFraud'] == 1).sum() / len(self.df_data) * 100:.4f}%)")
        print(f"\nTransaction types:")
        print(self.df_data['type'].value_counts())
        print(f"\nAmount statistics:")
        print(self.df_data['amount'].describe())
        print(f"\nFeatures: {list(self.df_data.columns)}")

    def save_to_s3(self, str_filename):
        """Save cleaned data as parquet directly to S3."""
        if self.df_data is None:
            raise ValueError("Data not loaded.")
        
        str_s3_key = f'00_data_collection/{str_filename}'
        print(f"\nSaving to S3: s3://{self.str_bucket}/{str_s3_key}")
        
        # Write parquet to buffer and upload directly to S3
        from io import BytesIO
        buffer = BytesIO()
        self.df_data.to_parquet(buffer, index=False)
        buffer.seek(0)
        
        try:
            self.s3_client.put_object(
                Bucket=self.str_bucket,
                Key=str_s3_key,
                Body=buffer.getvalue()
            )
            print(f"Successfully uploaded {str_filename} to S3")
        except Exception as e:
            print(f"Error uploading to S3: {e}")

## Constants

In [3]:
str_bucket = 'anomaly-detection-demo-repo'
str_task = 'data_collection'
str_dirname_output = './output'

## Output Directory

In [4]:
try:
    os.mkdir(str_dirname_output)
except FileExistsError:
    pass
print(f"Output directory ready: {str_dirname_output}")

Output directory ready: ./output


## Download and Prepare Data

In [5]:
# Initialize collector
collector = DataCollector(str_bucket, str_dirname_output)

# Download from Kaggle
df_raw = collector.download_from_kaggle()

Dataset downloaded to: /home/ec2-user/.cache/kagglehub/datasets/ealaxi/paysim1/versions/2
Loading CSV: /home/ec2-user/.cache/kagglehub/datasets/ealaxi/paysim1/versions/2/PS_20174392719_1491204439457_log.csv


In [6]:
# Display first few rows
print("First few rows of raw data:")
print(df_raw.head())
print(f"\nData shape: {df_raw.shape}")
print(f"\nColumn names and types:")
print(df_raw.dtypes)

First few rows of raw data:
   step      type    amount     nameOrig  oldbalanceOrg  newbalanceOrig  \
0     1   PAYMENT   9839.64  C1231006815       170136.0       160296.36   
1     1   PAYMENT   1864.28  C1666544295        21249.0        19384.72   
2     1  TRANSFER    181.00  C1305486145          181.0            0.00   
3     1  CASH_OUT    181.00   C840083671          181.0            0.00   
4     1   PAYMENT  11668.14  C2048537720        41554.0        29885.86   

      nameDest  oldbalanceDest  newbalanceDest  isFraud  isFlaggedFraud  
0  M1979787155             0.0             0.0        0               0  
1  M2044282225             0.0             0.0        0               0  
2   C553264065             0.0             0.0        1               0  
3    C38997010         21182.0             0.0        1               0  
4  M1230701703             0.0             0.0        0               0  

Data shape: (6362620, 11)

Column names and types:
step                int64

In [7]:
# Clean the data
df_cleaned = collector.clean_data()


Cleaning data...
Total null values: 0
Data shape after cleaning: (6362620, 11)


In [8]:
# Get summary statistics
collector.get_summary_stats()


=== SUMMARY STATISTICS ===
Total records: 6,362,620
Total features: 11

Fraud statistics:
  Non-fraud: 6,354,407 (99.87%)
  Fraud: 8,213 (0.1291%)

Transaction types:
type
CASH_OUT    2237500
PAYMENT     2151495
CASH_IN     1399284
TRANSFER     532909
DEBIT         41432
Name: count, dtype: int64

Amount statistics:
count    6.362620e+06
mean     1.798619e+05
std      6.038582e+05
min      0.000000e+00
25%      1.338957e+04
50%      7.487194e+04
75%      2.087215e+05
max      9.244552e+07
Name: amount, dtype: float64

Features: ['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud']


In [9]:
# Save to S3
collector.save_to_s3('paysim_clean.parquet')


Saving to S3: s3://anomaly-detection-demo-repo/00_data_collection/paysim_clean.parquet
Successfully uploaded paysim_clean.parquet to S3


## Completion

In [10]:
print("\n=== DATA COLLECTION COMPLETE ===")
print(f"File ready for next stage: s3://{str_bucket}/00_data_collection/paysim_clean.parquet")


=== DATA COLLECTION COMPLETE ===
File ready for next stage: s3://anomaly-detection-demo-repo/00_data_collection/paysim_clean.parquet
